In [ ]:
!pip install nibabel scikit-image tqdm matplotlib --quiet
!pip install SimpleITK --quiet   # for proper voxel-spacing-aware resampling

In [ ]:
import os, gc
import numpy as np
import nibabel as nib
import SimpleITK as sitk
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from tqdm import tqdm
from scipy.ndimage import zoom, rotate
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model
from tensorflow.keras import backend as K

print("TF version:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

In [ ]:
DATASET   = Path("/kaggle/input/datasets/awsaf49/brats20-dataset-training-validation")
TRAIN_DIR = DATASET / "BraTS2020_TrainingData" / "MICCAI_BraTS2020_TrainingData"
OUT_IMAGES = Path("/kaggle/working/images")
OUT_MASKS  = Path("/kaggle/working/masks")
OUT_META   = Path("/kaggle/working/meta")   # NEW — stores voxel spacing per patient
OUT_IMAGES.mkdir(exist_ok=True)
OUT_MASKS.mkdir(exist_ok=True)
OUT_META.mkdir(exist_ok=True)

MODALITIES = ["t1", "t1ce", "t2", "flair"]
IMG_SIZE   = 128
DEPTH      = 128

In [ ]:
def load_nifti_with_spacing(path):
    """Returns (volume float32, voxel_spacing tuple)"""
    img    = nib.load(path)
    volume = img.get_fdata(dtype=np.float32)
    # voxel dimensions in mm — critical for correct volume computation later
    spacing = img.header.get_zooms()[:3]
    return volume, spacing

In [ ]:
def normalize_zscore(volume):
    """
    Z-score normalize using only brain voxels (non-zero mask).
    MinMax is unstable across patients — one bright artifact makes the
    whole scan map incorrectly. Z-score on brain-only voxels is the
    BraTS standard.
    """
    brain_mask = volume > 0
    if brain_mask.sum() == 0:
        return volume
    mean = volume[brain_mask].mean()
    std  = volume[brain_mask].std()
    if std < 1e-8:
        return volume
    volume = (volume - mean) / (std + 1e-8)
    volume[~brain_mask] = 0.0   # keep background as 0
    return volume

In [ ]:
def crop_brain(volume, pad=5):
    """Crop to tight bounding box around non-background voxels."""
    mask   = volume > np.percentile(volume[volume > 0], 1) if (volume > 0).any() else volume > 0
    coords = np.where(mask)
    if len(coords[0]) == 0:
        return volume
    x0,x1 = max(coords[0].min()-pad,0), coords[0].max()+pad
    y0,y1 = max(coords[1].min()-pad,0), coords[1].max()+pad
    z0,z1 = max(coords[2].min()-pad,0), coords[2].max()+pad
    return volume[x0:x1, y0:y1, z0:z1]

In [ ]:
def resample_volume(volume, original_shape, order=1):
    """
    Resample to (IMG_SIZE, IMG_SIZE, DEPTH).
    Returns (resampled_volume, zoom_factors) — zoom factors needed to
    convert voxel count back to physical volume.
    """
    x, y, z    = original_shape
    zoom_factors = (IMG_SIZE/x, IMG_SIZE/y, DEPTH/z)
    resampled    = zoom(volume, zoom_factors, order=order)
    return resampled, zoom_factors

In [ ]:
def remap_labels(mask):
    """
    BraTS labels:  0=background, 1=NCR/NET, 2=ED, 4=ET
    We remap 4→3 so we get contiguous 0,1,2,3 for softmax,
    but we also store a note of the mapping so evaluation can
    reconstruct WT/TC/ET composite regions.

    WT (Whole Tumor)    = labels 1+2+3  (all tumor)
    TC (Tumor Core)     = labels 1+3    (NCR + ET)
    ET (Enhancing Tumor)= label  3      (ET only)
    """
    mask = mask.copy()
    mask[mask == 4] = 3
    return mask.astype(np.uint8)

In [ ]:
def process_patient(patient_folder):
    patient_id = patient_folder.name
    volumes    = []
    spacings   = []

    for mod in MODALITIES:
        path = patient_folder / f"{patient_id}_{mod}.nii"
        if not path.exists():
            path = path.with_suffix(".nii.gz")

        vol, spacing = load_nifti_with_spacing(path)
        spacings.append(spacing)

        orig_shape = vol.shape
        vol = crop_brain(vol)
        vol, zoom_factors = resample_volume(vol, vol.shape)
        vol = normalize_zscore(vol)          # FIX #2
        volumes.append(vol)

    image = np.stack(volumes, axis=-1).astype(np.float16)

    # Load mask — use nearest-neighbour (order=0) to avoid interpolating labels
    mask_path = patient_folder / f"{patient_id}_seg.nii"
    if not mask_path.exists():
        mask_path = mask_path.with_suffix(".nii.gz")

    mask, mask_spacing = load_nifti_with_spacing(mask_path)
    orig_mask_shape    = mask.shape
    mask = crop_brain(mask)
    mask, mzoom = resample_volume(mask, mask.shape, order=0)  # order=0 = no label blending
    mask = remap_labels(mask)

    # Compute true voxel volume in mm³ (FIX #3)
    # Original spacing * zoom_factor gives spacing in resampled space
    voxel_spacing_mm = tuple(s * (1/z) for s, z in zip(spacings[0], zoom_factors))
    voxel_vol_mm3    = float(np.prod(voxel_spacing_mm))

    return image, mask, voxel_vol_mm3

In [ ]:
patients = sorted([p for p in TRAIN_DIR.glob("*") if p.is_dir()])
print(f"Found {len(patients)} patients")

failed = []
for patient in tqdm(patients):
    try:
        image, mask, voxel_vol = process_patient(patient)
        pid = patient.name
        np.save(OUT_IMAGES / f"{pid}.npy", image)
        np.save(OUT_MASKS  / f"{pid}.npy", mask)
        np.save(OUT_META   / f"{pid}_voxel.npy", np.array([voxel_vol]))  # save real voxel size
    except Exception as e:
        failed.append((patient.name, str(e)))

print(f"\nDone. Failed: {len(failed)}")
for f in failed:
    print(f)

In [ ]:
image_paths = sorted(list(OUT_IMAGES.glob("*.npy")))
mask_paths  = sorted(list(OUT_MASKS.glob("*.npy")))
meta_paths  = sorted(list(OUT_META.glob("*_voxel.npy")))

# Sanity check — all three lists must align by patient ID
img_ids  = [p.stem for p in image_paths]
msk_ids  = [p.stem for p in mask_paths]
assert img_ids == msk_ids, "Image/mask mismatch — check OUT_IMAGES and OUT_MASKS"

train_imgs, val_imgs, train_msks, val_msks = train_test_split(
    image_paths, mask_paths,
    test_size=0.2, random_state=42
)

print(f"Train: {len(train_imgs)} | Val: {len(val_imgs)}")

In [ ]:
def random_flip_3d(image, mask):
    """Randomly flip along any of the 3 spatial axes."""
    for axis in [0, 1, 2]:
        if np.random.rand() > 0.5:
            image = np.flip(image, axis=axis).copy()
            mask  = np.flip(mask,  axis=axis).copy()
    return image, mask


def random_rotate_3d(image, mask, max_angle=15):
    """
    Rotate in the axial plane (around z-axis) by a random angle.
    Uses order=1 for image (bilinear) and order=0 for mask (nearest).
    Keeps labels integer — no blending artifacts.
    """
    angle = np.random.uniform(-max_angle, max_angle)
    rotated_image = np.stack([
        rotate(image[..., c], angle, axes=(0, 1),
               reshape=False, order=1, mode='constant', cval=0.0)
        for c in range(image.shape[-1])
    ], axis=-1)
    rotated_mask = rotate(mask, angle, axes=(0, 1),
                          reshape=False, order=0, mode='constant', cval=0)
    return rotated_image, rotated_mask.astype(np.uint8)


def random_intensity_jitter(image, sigma=0.1):
    """
    Add per-modality Gaussian noise + random brightness shift.
    Only applied to image, never to mask.
    Simulates scanner variability across institutions.
    """
    for c in range(image.shape[-1]):
        if np.random.rand() > 0.5:
            noise = np.random.normal(0, sigma, image[..., c].shape).astype(np.float32)
            image[..., c] = image[..., c] + noise
        if np.random.rand() > 0.5:
            shift = np.random.uniform(-0.1, 0.1)
            image[..., c] = image[..., c] + shift
    return image


def augment(image, mask):
    """
    Apply all augmentations in sequence.
    Only called during training — validation always gets clean data.
    """
    image, mask = random_flip_3d(image, mask)
    image, mask = random_rotate_3d(image, mask, max_angle=15)
    image       = random_intensity_jitter(image, sigma=0.1)
    return image, mask

In [ ]:
class BraTSGenerator(tf.keras.utils.Sequence):
    """
    Loads preprocessed .npy pairs, applies augmentation on training set,
    and returns one-hot encoded masks.

    Shapes returned:
      X : (batch, 128, 128, 128, 4)   — float32
      Y : (batch, 128, 128, 128, 4)   — float32 one-hot (4 classes)
    """

    def __init__(self, image_paths, mask_paths,
                 batch_size=2, augment=False, shuffle=True):
        self.image_paths = list(image_paths)
        self.mask_paths  = list(mask_paths)
        self.batch_size  = batch_size
        self.do_augment  = augment
        self.shuffle     = shuffle
        self.indices     = np.arange(len(self.image_paths))
        self.on_epoch_end()

    def __len__(self):
        return len(self.image_paths) // self.batch_size

    def on_epoch_end(self):
        """Shuffle at the end of every epoch (training only)."""
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __getitem__(self, idx):
        batch_idx = self.indices[idx * self.batch_size : (idx+1) * self.batch_size]

        X_batch, Y_batch = [], []

        for i in batch_idx:
            image = np.load(self.image_paths[i]).astype(np.float32)
            mask  = np.load(self.mask_paths[i])

            if self.do_augment:
                image, mask = augment(image, mask)

            # One-hot encode mask → (128, 128, 128, 4)
            mask_ohe = tf.keras.utils.to_categorical(mask, num_classes=4).astype(np.float32)

            X_batch.append(image)
            Y_batch.append(mask_ohe)

        return np.array(X_batch), np.array(Y_batch)

In [ ]:
train_gen = BraTSGenerator(
    train_imgs, train_msks,
    batch_size=2,
    augment=True,
    shuffle=True
)

val_gen = BraTSGenerator(
    val_imgs, val_msks,
    batch_size=2,
    augment=False,
    shuffle=False
)

print(f"Train batches: {len(train_gen)} | Val batches: {len(val_gen)}")

# Quick shape check
x_sample, y_sample = train_gen[0]
print(f"X shape: {x_sample.shape} | dtype: {x_sample.dtype}")
print(f"Y shape: {y_sample.shape} | dtype: {y_sample.dtype}")
print(f"X value range: [{x_sample.min():.3f}, {x_sample.max():.3f}]")
print(f"Unique mask classes in batch: {np.unique(np.argmax(y_sample[0], axis=-1))}")

In [ ]:
# Class weights based on inverse frequency in BraTS2020
# Background ~82%, ED ~10%, NCR ~5%, ET ~3%
# We heavily upweight ET and TC to force the model to learn them
CLASS_WEIGHTS = tf.constant([0.1, 1.0, 1.0, 4.0], dtype=tf.float32)

def dice_coef(y_true, y_pred, smooth=1e-6):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    intersection = tf.reduce_sum(y_true * y_pred, axis=[1,2,3,4])
    union        = tf.reduce_sum(y_true, axis=[1,2,3,4]) + tf.reduce_sum(y_pred, axis=[1,2,3,4])
    return tf.reduce_mean((2. * intersection + smooth) / (union + smooth))

def dice_loss(y_true, y_pred, smooth=1e-6):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    total  = 0.0
    # Compute Dice per class separately, then weight
    for c, w in enumerate([0.1, 1.0, 1.0, 4.0]):
        yt = y_true[..., c]
        yp = y_pred[..., c]
        intersection = tf.reduce_sum(yt * yp)
        union        = tf.reduce_sum(yt) + tf.reduce_sum(yp)
        class_dice   = (2. * intersection + smooth) / (union + smooth)
        total        += w * (1.0 - class_dice)
    return total / 6.1   # normalize by sum of weights

def focal_loss(y_true, y_pred, gamma=3.0, alpha=0.25):
    """
    gamma raised to 3.0 (from 2.0) — harder focus on difficult ET voxels.
    """
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
    ce     = -y_true * tf.math.log(y_pred)
    weight = alpha * y_true * tf.pow(1.0 - y_pred, gamma)
    # Apply class weights so ET focal loss is amplified 4x
    class_w = tf.reshape(CLASS_WEIGHTS, [1,1,1,1,4])
    return tf.reduce_mean(tf.reduce_sum(class_w * weight * ce, axis=-1))

def combined_loss(y_true, y_pred):
    return 0.5 * dice_loss(y_true, y_pred) + 0.5 * focal_loss(y_true, y_pred)

In [ ]:
# These are the 3 official BraTS evaluation metrics
# Your previous notebook only computed a single global Dice — this is the fix

def dice_wt(y_true, y_pred):
    """Whole Tumor — all labels > 0 (classes 1+2+3)"""
    yt = tf.cast(tf.reduce_sum(y_true[..., 1:], axis=-1) > 0, tf.float32)
    yp = tf.cast(tf.reduce_sum(y_pred[..., 1:], axis=-1) > 0.5, tf.float32)
    i  = tf.reduce_sum(yt * yp)
    return (2.*i + 1e-6) / (tf.reduce_sum(yt) + tf.reduce_sum(yp) + 1e-6)

def dice_tc(y_true, y_pred):
    """Tumor Core — NCR(1) + ET(3), excludes edema"""
    yt = tf.cast((y_true[...,1] + y_true[...,3]) > 0, tf.float32)
    yp = tf.cast((y_pred[...,1] + y_pred[...,3]) > 0.5, tf.float32)
    i  = tf.reduce_sum(yt * yp)
    return (2.*i + 1e-6) / (tf.reduce_sum(yt) + tf.reduce_sum(yp) + 1e-6)

def dice_et(y_true, y_pred):
    """Enhancing Tumor — class 3 only (hardest region)"""
    yt = y_true[..., 3]
    yp = tf.cast(y_pred[..., 3] > 0.5, tf.float32)
    i  = tf.reduce_sum(yt * yp)
    return (2.*i + 1e-6) / (tf.reduce_sum(yt) + tf.reduce_sum(yp) + 1e-6)

In [ ]:
def residual_block(x, filters):
    """
    Two Conv3D layers with BatchNorm + ReLU, plus a residual shortcut.
    The shortcut learns to add identity information back, preventing
    gradient vanishing in deep 3D networks.
    """
    shortcut = Conv3D(filters, 1, padding='same')(x)
    shortcut = BatchNormalization()(shortcut)

    x = Conv3D(filters, 3, padding='same')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)

    x = Conv3D(filters, 3, padding='same')(x)
    x = BatchNormalization()(x)

    x = Add()([x, shortcut])
    x = Activation('relu')(x)
    return x


def attention_gate(g, s, filters):
    """
    Attention gate on skip connections.
    g = gating signal from decoder (coarser, more semantic)
    s = skip connection from encoder (finer, more spatial)

    Learns to suppress irrelevant spatial regions in the skip
    connection before concatenating — so the decoder focuses on
    tumor-relevant features rather than everything the encoder saw.
    """
    Wg = Conv3D(filters, 1, padding='same')(g)
    Wg = BatchNormalization()(Wg)

    Ws = Conv3D(filters, 1, padding='same')(s)
    Ws = BatchNormalization()(Ws)

    psi = Activation('relu')(Add()([Wg, Ws]))
    psi = Conv3D(1, 1, padding='same')(psi)
    psi = BatchNormalization()(psi)
    psi = Activation('sigmoid')(psi)   # attention map: 0=ignore, 1=attend

    return Multiply()([s, psi])        # gate the skip connection

In [ ]:
def build_attention_unet(input_shape=(128, 128, 128, 4), num_classes=4):

    inputs = Input(input_shape)

    # ── Encoder ──────────────────────────────────────────────────
    c1 = residual_block(inputs, 32)
    p1 = MaxPooling3D()(c1)             # 64³

    c2 = residual_block(p1, 64)
    p2 = MaxPooling3D()(c2)             # 32³

    c3 = residual_block(p2, 128)
    p3 = MaxPooling3D()(c3)             # 16³

    # ── Bottleneck ────────────────────────────────────────────────
    bn = residual_block(p3, 256)        # 16³, 256 filters

    # ── Decoder with attention gates ──────────────────────────────
    # Block 1: 16³ → 32³
    u1 = Conv3DTranspose(128, 2, strides=2, padding='same')(bn)
    a1 = attention_gate(g=u1, s=c3, filters=64)
    u1 = Concatenate()([u1, a1])
    c4 = residual_block(u1, 128)

    # Block 2: 32³ → 64³
    u2 = Conv3DTranspose(64, 2, strides=2, padding='same')(c4)
    a2 = attention_gate(g=u2, s=c2, filters=32)
    u2 = Concatenate()([u2, a2])
    c5 = residual_block(u2, 64)

    # Block 3: 64³ → 128³
    u3 = Conv3DTranspose(32, 2, strides=2, padding='same')(c5)
    a3 = attention_gate(g=u3, s=c1, filters=16)
    u3 = Concatenate()([u3, a3])
    c6 = residual_block(u3, 32)

    # ── Output ────────────────────────────────────────────────────
    outputs = Conv3D(num_classes, 1, activation='softmax')(c6)

    return Model(inputs, outputs, name='AttentionUNet3D')


model = build_attention_unet()
model.summary()

In [ ]:
# Reset weights — start fresh with the new loss
model = build_attention_unet()

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=combined_loss,
    metrics=[dice_coef, dice_wt, dice_tc, dice_et]
)

print("Parameters:", f"{model.count_params():,}")

In [ ]:
from tensorflow.keras.callbacks import (
    ModelCheckpoint, ReduceLROnPlateau,
    EarlyStopping, CSVLogger
)

callbacks = [
    ModelCheckpoint(
        "best_attention_unet_v2.keras",
        monitor='val_dice_wt',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_dice_wt',
        factor=0.5,
        patience=10,
        min_lr=1e-7,
        verbose=1
    ),
    EarlyStopping(
        monitor='val_dice_wt',
        patience=20,
        mode='max',
        restore_best_weights=True,
        verbose=1
    ),
    CSVLogger("training_log_v2.csv")
]

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=100,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
from tensorflow.keras.callbacks import (
    ModelCheckpoint, ReduceLROnPlateau,
    EarlyStopping, CSVLogger
)

callbacks = [
    ModelCheckpoint(
        "best_attention_unet.keras",
        monitor='val_dice_wt',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_dice_wt',
        factor=0.5,
        patience=10,
        min_lr=1e-7,
        verbose=1
    ),
    EarlyStopping(
        monitor='val_dice_wt',
        patience=20,
        mode='max',
        restore_best_weights=True,
        verbose=1
    ),
    CSVLogger("training_log.csv")
]

EPOCHS = 100

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
from tensorflow.keras.models import load_model

# Custom objects needed to deserialize the saved model
model = load_model(
    "/kaggle/working/best_attention_unet_v2.keras",
    custom_objects={
        "combined_loss": combined_loss,
        "dice_coef":     dice_coef,
        "dice_wt":       dice_wt,
        "dice_tc":       dice_tc,
        "dice_et":       dice_et,
    }
)
print("Model loaded.")

# ── Preprocessing (mirrors training pipeline exactly) ──────────────
def load_and_preprocess_patient(patient_folder):
    """
    Load a raw BraTS patient folder, apply the same crop/resample/
    normalize pipeline used during training.
    Returns:
        volume      : (128,128,128,4) float32  — model input
        voxel_vol   : float  — mm³ per voxel in resampled space
        patient_id  : str
    """
    patient_folder = Path(patient_folder)
    patient_id     = patient_folder.name
    volumes, spacings = [], []

    for mod in MODALITIES:
        path = patient_folder / f"{patient_id}_{mod}.nii"
        if not path.exists():
            path = path.with_suffix(".nii.gz")

        img      = nib.load(path)
        vol      = img.get_fdata(dtype=np.float32)
        spacing  = img.header.get_zooms()[:3]
        spacings.append(spacing)

        orig_shape  = vol.shape
        vol         = crop_brain(vol)
        vol, zf     = resample_volume(vol, vol.shape)
        vol         = normalize_zscore(vol)
        volumes.append(vol)

    image = np.stack(volumes, axis=-1).astype(np.float32)

    # True voxel volume in resampled space
    voxel_spacing_mm = tuple(s * (1/z) for s, z in zip(spacings[0], zf))
    voxel_vol_mm3    = float(np.prod(voxel_spacing_mm))

    return image, voxel_vol_mm3, patient_id

In [ ]:
def predict_patient(patient_folder):
    """
    Full inference pipeline for one patient.
    Returns:
        volume   : (128,128,128,4) float32  — preprocessed MRI
        pred_mask: (128,128,128)   uint8    — argmax predicted labels
        prob_map : (128,128,128,4) float32  — raw softmax probabilities
        voxel_vol: float — mm³ per resampled voxel
        pid      : str
    """
    volume, voxel_vol, pid = load_and_preprocess_patient(patient_folder)
    inp       = np.expand_dims(volume, axis=0)        # (1,128,128,128,4)
    prob_map  = model.predict(inp, verbose=0)[0]      # (128,128,128,4)
    pred_mask = np.argmax(prob_map, axis=-1).astype(np.uint8)
    return volume, pred_mask, prob_map, voxel_vol, pid

Val_DIR = DATASET / "BraTS2020_ValidationData" / "MICCAI_BraTS2020_ValidationData"

# ── Test on patient 001 ────────────────────────────────────────────
sample_patient = Val_DIR / "BraTS20_Validation_005"
volume, pred_mask, prob_map, voxel_vol, pid = predict_patient(sample_patient)

print(f"Patient       : {pid}")
print(f"Volume shape  : {volume.shape}")
print(f"Mask shape    : {pred_mask.shape}")
print(f"Unique labels : {np.unique(pred_mask)}")
print(f"Voxel volume  : {voxel_vol:.4f} mm³")
print(f"Tumor voxels  : {np.sum(pred_mask > 0):,}")

In [ ]:
def compute_tumor_metrics(pred_mask, voxel_vol_mm3):
    """
    Computes physical tumor measurements using true voxel spacing.
    Your previous notebook assumed 1 voxel = 1mm³ — this uses the
    real resampled voxel size so the numbers are clinically meaningful.
    """
    metrics = {}

    # ── Per sub-region voxel counts ───────────────────────────────
    ncr_vox = np.sum(pred_mask == 1)   # Necrotic core
    ed_vox  = np.sum(pred_mask == 2)   # Edema
    et_vox  = np.sum(pred_mask == 3)   # Enhancing tumor
    wt_vox  = np.sum(pred_mask > 0)    # Whole tumor

    # ── Convert to physical volume ────────────────────────────────
    metrics["ncr_volume_cm3"] = round(ncr_vox * voxel_vol_mm3 / 1000, 2)
    metrics["ed_volume_cm3"]  = round(ed_vox  * voxel_vol_mm3 / 1000, 2)
    metrics["et_volume_cm3"]  = round(et_vox  * voxel_vol_mm3 / 1000, 2)
    metrics["wt_volume_cm3"]  = round(wt_vox  * voxel_vol_mm3 / 1000, 2)

    # ── 3D bounding box & diameter ───────────────────────────────
    coords = np.where(pred_mask > 0)
    if len(coords[0]) > 0:
        z0,z1 = coords[0].min(), coords[0].max()
        y0,y1 = coords[1].min(), coords[1].max()
        x0,x1 = coords[2].min(), coords[2].max()
        metrics["bbox"]             = (z0,z1,y0,y1,x0,x1)
        metrics["diameter_mm"]      = round(max(z1-z0, y1-y0, x1-x0) * (voxel_vol_mm3**(1/3)), 1)
        metrics["centroid"]         = (
            int((z0+z1)/2), int((y0+y1)/2), int((x0+x1)/2)
        )
    else:
        metrics["bbox"]        = None
        metrics["diameter_mm"] = 0
        metrics["centroid"]    = (64, 64, 64)

    return metrics


metrics = compute_tumor_metrics(pred_mask, voxel_vol)
print(f"\n── Tumor Metrics for {pid} ──")
print(f"Whole tumor  : {metrics['wt_volume_cm3']} cm³")
print(f"Necrotic core: {metrics['ncr_volume_cm3']} cm³")
print(f"Edema        : {metrics['ed_volume_cm3']} cm³")
print(f"Enhancing    : {metrics['et_volume_cm3']} cm³")
print(f"Max diameter : {metrics['diameter_mm']} mm")
print(f"Centroid     : {metrics['centroid']}")

In [ ]:
def build_lobe_atlas(shape=(128, 128, 128)):
    """
    Builds a simple MNI-space approximation of the 4 cerebral lobes
    at 128³ resolution. Based on standard anatomical landmarks
    mapped proportionally to the cropped/resampled brain volume.

    Axes in BraTS resampled space:
        axis 0 (z) = inferior → superior
        axis 1 (y) = posterior → anterior
        axis 2 (x) = right → left

    Returns a uint8 atlas where:
        1 = Frontal lobe
        2 = Parietal lobe
        3 = Temporal lobe
        4 = Occipital lobe
        0 = Other (brainstem, cerebellum, background)

    FIX: Initialise with 1 (Frontal) instead of 0 so every voxel
    belongs to a lobe from the start — no silent "Other" sink that
    swallows tumor voxels and breaks the 100 % sum.
    """
    D, H, W = shape

    # ── KEY FIX: fill everything as Frontal first ─────────────────
    # Previously np.zeros left large regions as label-0 ("Other"),
    # causing ~27 % of tumor voxels to vanish from the percentage sum.
    atlas = np.ones(shape, dtype=np.uint8)   # <── was np.zeros

    # ── Proportional boundaries ───────────────────────────────────
    z_mid  = int(D * 0.50)   # inferior/superior split (50 %)
    z_sup  = int(D * 0.72)   # high-superior threshold (72 %)

    y_front_end = int(H * 0.72)   # anterior edge of frontal
    y_par_start = int(H * 0.38)   # frontal/parietal boundary
    y_occ_start = int(H * 0.18)   # parietal/occipital boundary

    z_temp_max  = int(D * 0.62)   # temporal only in inferior 62 %
    y_temp_end  = int(H * 0.62)   # temporal only in anterior 62 %

    # ── Parietal: superior mid-posterior ──────────────────────────
    # High superior band
    atlas[z_sup:,      y_occ_start:y_par_start, :] = 2
    # Mid superior band
    atlas[z_mid:z_sup, int(H * 0.25):y_par_start, :] = 2

    # ── Occipital: posterior pole (applied before Temporal so
    #    Temporal can override the inferior-posterior overlap) ─────
    atlas[:,         :y_occ_start, :] = 4

    # ── Temporal: inferior lateral ────────────────────────────────
    # Overrides whatever was assigned in the inferior-anterior block
    atlas[:z_temp_max, :y_temp_end, :] = 3

    # ── Re-assert Occipital at posterior pole (highest priority) ──
    # Temporal assignment above may have overwritten the posterior-
    # inferior corner; stamp occipital back on top.
    atlas[:,         :y_occ_start, :] = 4

    return atlas


lobe_atlas = build_lobe_atlas(shape=(128, 128, 128))
lobe_names = {0: "Other", 1: "Frontal", 2: "Parietal", 3: "Temporal", 4: "Occipital"}

# ── Verify: Other should now be 0 voxels ──────────────────────────
total_voxels = 128 ** 3
print(f"Total voxels : {total_voxels:,}")
print()
for k, v in lobe_names.items():
    count = np.sum(lobe_atlas == k)
    pct   = 100.0 * count / total_voxels
    print(f"{v:12s}: {count:>8,} voxels  ({pct:5.1f}%)")

# Sanity check
other_count = np.sum(lobe_atlas == 0)
assert other_count == 0, f"❌ Still {other_count:,} unlabelled voxels — check boundary logic!"
print("\n✅ Atlas is gap-free. All voxels assigned to a lobe.")


In [ ]:
def compute_lobe_involvement(pred_mask, lobe_atlas):
    """
    For each brain lobe, compute what percentage of the tumor
    (whole tumor mask) falls within that lobe.

    Returns a dict with lobe name → percentage of tumor volume.

    FIX 1: Removed the `if lobe_name == "Other": continue` guard —
            with the gap-free atlas, Other will always be 0 %, but
            including it lets us assert the sum and catch future bugs.
    FIX 2: Added a 100 % sanity assertion so any future atlas change
            that re-introduces gaps surfaces immediately.
    """
    tumor_mask  = pred_mask > 0
    tumor_total = int(tumor_mask.sum())

    # Edge case: no tumor detected at all
    if tumor_total == 0:
        return {name: 0.0 for name in lobe_names.values()}

    involvement = {}
    for lobe_id, lobe_name in lobe_names.items():
        lobe_region = lobe_atlas == lobe_id
        overlap     = int(np.logical_and(tumor_mask, lobe_region).sum())
        pct         = round(100.0 * overlap / tumor_total, 1)
        involvement[lobe_name] = pct

    # ── Sanity check: all lobe percentages must sum to ~100 % ─────
    # Rounding across 5 buckets can drift by ±0.5 at most, so
    # anything outside [99.0, 101.0] means the atlas still has gaps.
    total_pct = sum(involvement.values())
    assert 99.0 <= total_pct <= 101.0, (
        f"❌ Lobe percentages sum to {total_pct:.1f}% — atlas has unlabelled voxels! "
        f"Run build_lobe_atlas() again and check the 'Other' voxel count."
    )

    # ── Sort: named lobes by involvement (desc), Other always last ─
    named = {k: v for k, v in involvement.items() if k != "Other"}
    named = dict(sorted(named.items(), key=lambda x: x[1], reverse=True))
    named["Other"] = involvement["Other"]   # always visible for auditing

    return named


# ── Run & display ─────────────────────────────────────────────────
lobe_inv = compute_lobe_involvement(pred_mask, lobe_atlas)

print(f"\n── Lobe Involvement for {pid} ──")
for lobe, pct in lobe_inv.items():
    bar = "█" * int(pct / 2)
    print(f"{lobe:12s}: {pct:5.1f}%  {bar}")

# Explicit sum line so it's always visible in output
print(f"\n{'Total':12s}: {sum(lobe_inv.values()):5.1f}%")

In [ ]:
# Label colors: 0=background(transparent), 1=NCR(red), 2=ED(yellow), 3=ET(cyan)
LABEL_COLORS = {
    1: (255,  80,  80),   # NCR — red
    2: (255, 220,  50),   # ED  — yellow
    3: ( 80, 220, 255),   # ET  — cyan
}
LABEL_NAMES = {1:"NCR/NET", 2:"Edema", 3:"Enhancing"}

def annotate_slice(mri_slice, mask_slice, alpha=0.45):
    """
    Overlay colored tumor labels on a grayscale MRI slice.
    mri_slice : (H,W) float   — single modality (FLAIR recommended)
    mask_slice: (H,W) uint8   — predicted labels 0-3
    Returns   : (H,W,3) uint8 — RGB annotated image
    """
    # Normalize MRI slice to 0-255
    mn, mx = mri_slice.min(), mri_slice.max()
    if mx > mn:
        gray = ((mri_slice - mn) / (mx - mn) * 255).astype(np.uint8)
    else:
        gray = np.zeros_like(mri_slice, dtype=np.uint8)

    rgb = np.stack([gray, gray, gray], axis=-1).astype(np.float32)

    for label_id, color in LABEL_COLORS.items():
        region = mask_slice == label_id
        if region.any():
            for c_idx, c_val in enumerate(color):
                rgb[..., c_idx] = np.where(
                    region,
                    (1 - alpha) * rgb[..., c_idx] + alpha * c_val,
                    rgb[..., c_idx]
                )

    return np.clip(rgb, 0, 255).astype(np.uint8)


def export_annotated_slices(volume, pred_mask, metrics, lobe_inv,
                             patient_id, out_dir="/kaggle/working/annotated"):
    """
    Exports annotated PNG for every axial slice that contains tumor,
    plus a summary grid of axial/coronal/sagittal views through centroid.
    """
    out_path = Path(out_dir) / patient_id
    out_path.mkdir(parents=True, exist_ok=True)

    flair    = volume[..., 3]    # FLAIR channel — best for tumor boundary
    centroid = metrics["centroid"]

    # ── 1. Export every axial slice containing tumor ───────────────
    tumor_slices = [z for z in range(pred_mask.shape[0])
                    if np.any(pred_mask[z, :, :] > 0)]

    for z in tumor_slices:
        img = annotate_slice(flair[z, :, :], pred_mask[z, :, :])

        fig, ax = plt.subplots(1, 1, figsize=(4, 4), dpi=100)
        ax.imshow(img)
        ax.set_title(f"{patient_id} | Axial z={z}", fontsize=8, pad=3)
        ax.axis("off")

        # Patch legend
        patches = [mpatches.Patch(color=np.array(c)/255, label=n)
                   for lbl,(c,n) in
                   zip(LABEL_COLORS.keys(),
                       zip(LABEL_COLORS.values(), LABEL_NAMES.values()))
                   if np.any(pred_mask[z,:,:] == lbl)]
        if patches:
            ax.legend(handles=patches, loc="lower right",
                      fontsize=6, framealpha=0.6)

        plt.tight_layout(pad=0.3)
        plt.savefig(out_path / f"axial_z{z:03d}.png",
                    bbox_inches="tight", dpi=100)
        plt.close()

    # ── 2. Summary 3-plane figure through centroid ─────────────────
    cz, cy, cx = centroid
    fig, axes  = plt.subplots(1, 3, figsize=(14, 5), dpi=120)
    fig.patch.set_facecolor("#111111")

    planes = [
        (annotate_slice(flair[cz,:,:],  pred_mask[cz,:,:]),  f"Axial z={cz}"),
        (annotate_slice(flair[:,cy,:],  pred_mask[:,cy,:]),   f"Coronal y={cy}"),
        (annotate_slice(flair[:,:,cx],  pred_mask[:,:,cx]),   f"Sagittal x={cx}"),
    ]

    for ax, (img, title) in zip(axes, planes):
        ax.imshow(img)
        ax.set_title(title, color="white", fontsize=10)
        ax.axis("off")

    # Lobe bar chart inset
    ax_lobe = fig.add_axes([0.01, 0.02, 0.28, 0.22])
    lobes   = list(lobe_inv.keys())
    pcts    = list(lobe_inv.values())
    colors  = ["#4CA3DD","#E86A3A","#5CC85C","#B05AC4"][:len(lobes)]
    ax_lobe.barh(lobes, pcts, color=colors, height=0.5)
    ax_lobe.set_xlim(0, 100)
    ax_lobe.set_xlabel("% of tumor", color="white", fontsize=7)
    ax_lobe.tick_params(colors="white", labelsize=7)
    ax_lobe.set_facecolor("#222222")
    for spine in ax_lobe.spines.values():
        spine.set_edgecolor("#555555")

    # Metrics text
    fig.text(0.36, 0.04,
        f"WT: {metrics['wt_volume_cm3']} cm³  |  "
        f"ET: {metrics['et_volume_cm3']} cm³  |  "
        f"NCR: {metrics['ncr_volume_cm3']} cm³  |  "
        f"ED: {metrics['ed_volume_cm3']} cm³  |  "
        f"Ø {metrics['diameter_mm']} mm",
        color="white", fontsize=9, va="bottom"
    )

    plt.suptitle(f"Brain Tumor Analysis — {patient_id}",
                 color="white", fontsize=13, y=1.01)
    plt.tight_layout()
    plt.savefig(out_path / "summary_3plane.png",
                bbox_inches="tight", facecolor="#111111", dpi=120)
    plt.close()

    print(f"Saved {len(tumor_slices)} axial slices + summary → {out_path}")
    return out_path


out_path = export_annotated_slices(
    volume, pred_mask, metrics, lobe_inv, pid
)

In [ ]:
# Show the summary figure right in the notebook
from IPython.display import Image as IPImage, display

summary_path = out_path / "summary_3plane.png"
display(IPImage(str(summary_path)))

print(f"\n── Final Report: {pid} ──────────────────────────")
print(f"Whole Tumor Volume : {metrics['wt_volume_cm3']} cm³")
print(f"Necrotic Core      : {metrics['ncr_volume_cm3']} cm³")
print(f"Edema              : {metrics['ed_volume_cm3']} cm³")
print(f"Enhancing Tumor    : {metrics['et_volume_cm3']} cm³")
print(f"Max Diameter       : {metrics['diameter_mm']} mm")
print(f"Centroid (z,y,x)   : {metrics['centroid']}")
print()
print("Lobe Involvement:")
for lobe, pct in lobe_inv.items():
    bar = "█" * int(pct / 2)
    print(f"  {lobe:12s}: {pct:5.1f}%  {bar}")

In [ ]:
"""import csv

results = []
val_patient_ids = [p.stem for p in val_imgs]   # patient IDs in val set

print(f"Running inference on {len(val_patient_ids)} validation patients...\n")

for pid_stem in tqdm(val_patient_ids):
    patient_folder = TRAIN_DIR / pid_stem
    if not patient_folder.exists():
        continue
    try:
        vol, pmask, pmap, vvol, pid = predict_patient(patient_folder)
        m   = compute_tumor_metrics(pmask, vvol)
        inv = compute_lobe_involvement(pmask, lobe_atlas)

        row = {
            "patient_id":      pid,
            "wt_volume_cm3":   m["wt_volume_cm3"],
            "ncr_volume_cm3":  m["ncr_volume_cm3"],
            "ed_volume_cm3":   m["ed_volume_cm3"],
            "et_volume_cm3":   m["et_volume_cm3"],
            "diameter_mm":     m["diameter_mm"],
            "centroid_z":      m["centroid"][0],
            "centroid_y":      m["centroid"][1],
            "centroid_x":      m["centroid"][2],
        }
        row.update({f"lobe_{k.lower()}": v for k,v in inv.items()})
        results.append(row)

    except Exception as e:
        print(f"  Skipped {pid_stem}: {e}")

# Save to CSV
csv_path = Path("/kaggle/working/tumor_analysis_report.csv")
if results:
    with open(csv_path, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=results[0].keys())
        writer.writeheader()
        writer.writerows(results)

print(f"\nDone. Report saved → {csv_path}")
print(f"Processed {len(results)} patients successfully.")

# Quick summary stats
import pandas as pd
df = pd.read_csv(csv_path)
print("\n── Cohort Summary ──────────────────────────────────")
print(df[["wt_volume_cm3","et_volume_cm3","diameter_mm"]].describe().round(2))"""
